# Revenue Mgmt Case Study (PySpark/SQL)

## Overview

This notebook performs a cruise line revenue management analysis using SQL on Databricks. The source data — revenue_mgmt_data_expmain.portfolio.revenue_mgmt_data_exp — contains cruise booking records originally in Spanish (columns like Fecha_Viaje, Barco, Compañia). The notebook follows a clear medallion-style pipeline: raw → cleaned → silver aggregation tables, then uses those tables for business analysis across three analytical lenses.

## Pipeline Architecture
_---The notebook been summarized with Genie AI---_

### 1. Data Wrangling 

The raw Spanish-columned table is cleaned in place via CREATE OR REPLACE TABLE, renaming all columns to English (e.g., Fecha_Viaje → sail_date, Barco → ship, Ingreso_Total_Reserva_USD → ntr). The dataset covers 8 cruise brands, each with one ship, sailing in 2018 – 2022.

### 2. Three Derived Tables

1. main.portfolio.raw_exec
- Grain: Executive-level (brand × ship × booking)
- Purpose: Adds computed pcd (passenger cruise days = sail_nights × pax_count × booked_cab) for rate calculations

2. main.portfolio.bookings
- Grain: Reservation-level
- Purpose: Clean booking records for granular analysis

3. main.portfolio.ship_detail
- Grain: Ship-level
- Purpose: Hardware specs (crew count, cabin capacity)


### 3. Silver Aggregation Tables 

Two silver tables enrich the data with time-based extractions and classifications:
1. main.portfolio.exec_silver 
- Aggregates raw_exec by brand, ship, sail date, booked date, product, cabin class, channel, etc. 
- Adds sail_qtr, WTS (weeks to sail), MTS (months to sail), and sail_night_group (Shorts / Getaway / Expedition).

2. main.portfolio.bookings_silver 
- Aggregates bookings similarly but at the reservation grain, adding pax counts and average load factor.

## Key Findings from the Analysis

### High-Level Brand KPIs 

All sailings in the dataset are from 2018. Across the 8 brands and 4 quarters: 
1. Celebrity Cruises is the revenue leader at $2.7 – $3.0M NTR per quarter, with the highest calculated APD ($167 – $182 per passenger per day). 
2. AIDA Cruises and Costa Cruceros are the lower-tier brands at $1.6–$1.8M NTR per quarter, with APDs around $123 – $143.

The notebook correctly calculates APD as SUM(ntr) / SUM(pcd) rather than averaging an average — a good analytical practice flagged in the comments. 

### Guest Ratings & Competitor Rate Comparison 

APD by cabin class (for MSC, Royal Caribbean, Disney): Disney commands the highest rates across all cabin types ($254 – $259), Royal Caribbean sits mid-range ($240 – $246), and MSC is the most affordable ($221 – $226). Notably, cabin class has minimal impact on rate within each brand — the brand itself is the primary price driver.

**Guest satisfaction scores by ship:**

Brand: Disney Cruise Line \
Ship: Dream \
Score: 4.9

Brand: Cunard \
Ship: Queen Mary 2 \
Score: 4.8

Brand: Royal Caribbean \
Ship: Icon of the Seas \
Score: 4.8

Brand: Celebrity Cruises \
Ship: Beyond \
Score: 4.7

Brand: MSC \
Ship: World Europa \
Score: 4.5

Brand: Norwegian \
Ship: Prima \
Score: 4.5

Brand: AIDA Cruises \
Ship: AIDAcosma \
Score: 4.3

Brand: Costa Cruises \
Ship: Smeralda \
Score: 4.2

### Booking Windows by Brand
All brands show a remarkably consistent booking curve: the bulk of bookings occur 2 – 3 months before sailing (24 – 28% of pax at MTS=2 or 3), with a long tail tapering off by 6 months. Last-minute bookings (MTS=0) account for only 4%, and bookings 7+ months out are negligible (less than 0.5%). This suggests a highly standardized booking pattern across the industry.

### Top Products by Brand 
Each brand specializes in a distinct route, and all carry remarkably similar passenger volumes (16.8 – 16.95K pax):

**Brand/Top Product**

AIDA Cruises
    - Canarias \
Celebrity Cruises
    - Alaska \
Costa Cruceros
    - Mediterráneo \
Cunard Line
    - Transatlántico \
Disney Cruise Line
    - Bahamas \
MSC Cruceros
    - Mediterráneo \
Norwegian Cruise Line
    - Fiordos \
Royal Caribbean
    - Caribe 

### Booking Type Breakdown 
Cunard's passengers favor All Inclusive (6,759 pax) over Full Board (4,078), with excursion-inclusive packages (All Inclusive + EXC: 3,413; Full Board + EXC: 2,551) being less popular — suggesting Cunard guests prefer simpler packages or book excursions independently.

### Historical Channel Trends 

From 2018 – 2021, channel distribution is remarkably stable: B2B dominates (14.7–14.8K pax/year), followed by Web (11.4–11.6K), then Direct (6.6–6.7K). In 2022, volumes collapse to roughly 10% of prior years (B2B: 1,703, Web: 1,243, Direct: 660) — this is likely either partial-year data or a COVID-related disruption effect.

### Ship Hardware: Cabin-to-Crew Ratio 
AIDAcosma has the highest ratio (1.8 cabins per crew member), while Disney Dream has the lowest (0.9) — meaning Disney has more crew per cabin, aligning with its premium service positioning and top guest rating (4.9). Most brands cluster around 1.1–1.2.

## Interpretation
This notebook demonstrates a well-structured cruise industry revenue management analysis with several strengths:

1. Correct KPI methodology — The author avoids the common pitfall of averaging averages for rate calculations, instead re-deriving APD from total revenue ÷ total passenger cruise days.
2. Medallion architecture — The raw → clean → silver table progression is clean and reusable, though the in-place CREATE OR REPLACE on the source table (Cell 8) is a notable design choice — it overwrites the original Spanish-columned data, making the transformation non-idempotent.

3. Consistent industry patterns — The data reveals highly uniform booking windows and passenger volumes across brands, suggesting the dataset may be synthetic or heavily normalized. The near-identical pax counts (~16.8–16.95K) across all brands' top products reinforce this.

4. One potential data issue — The 2022 channel trend drop-off warrants investigation (partial year vs. real disruption).

## Load and Preview Data

In [0]:
%sql
select * 
from main.portfolio.revenue_mgmt_data_exp

limit 100

RES_ID,Fecha_Viaje,Fecha_Reserva,Lead_Time_Dias,Noches_Estancia,Barco,Compañia,Tipo_Ruta,Suite_Type,Booking_Source,Package,Guest_Country,Cabinas_Totales_Barco,Cabinas_Reservadas,Porcentaje_Ocupacion_Ciclo,Pasajeros_Reserva,Tripulacion,Ingreso_Total_Reserva_USD,Gasto_Promedio_Diario_Huesped_USD,Puntuacion_Satisfaccion
RES-00001,2018-01-05,2017-08-18,140,6,MSC World Europa,MSC Cruceros,Mediterráneo,Balcony,Direct,All Inclusive,USA,2626,1,87.12,2,2138,2762.28,147.39,4.4
RES-00002,2018-01-05,2017-09-14,113,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,B2B,All Inclusive + EXC,Spain,2626,1,87.12,1,2138,1863.06,144.91,4.3
RES-00003,2018-01-05,2017-11-01,65,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Web,All Inclusive,Germany,2626,1,87.12,2,2138,2760.48,147.24,4.6
RES-00004,2018-01-05,2017-10-31,66,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,All Inclusive + EXC,USA,2626,2,87.12,2,2138,3883.68,158.04,4.3
RES-00005,2018-01-05,2017-09-17,110,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Direct,All Inclusive + EXC,USA,2626,1,87.12,2,2138,2723.28,144.14,4.3
RES-00006,2018-01-05,2017-11-21,45,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,All Inclusive,USA,2626,1,87.12,2,2138,2753.52,146.66,4.6
RES-00007,2018-01-05,2017-11-11,55,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,Full Board,UK,2626,2,87.12,2,2138,3671.4,140.35,4.6
RES-00008,2018-01-05,2017-09-13,114,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,B2B,All Inclusive,UK,2626,1,87.12,2,2138,2848.08,154.54,4.4
RES-00009,2018-01-05,2017-12-31,5,6,MSC World Europa,MSC Cruceros,Mediterráneo,Balcony,Direct,Full Board,Spain,2626,1,87.12,2,2138,2685.96,141.03,4.6
RES-00010,2018-01-05,2017-11-09,57,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Web,All Inclusive,Spain,2626,1,87.12,2,2138,2741.52,145.66,4.4



This result is stored as  and can be used in other Python and SQL cells.

In [0]:
display(_sqldf)

In [0]:
%sql
select * from _sqldf

## Data Wrangling 
--- since data is in different language, need to rename all columns at the least
--- also it'll probably be easier to store the specific datasets I need in a temp table/view within the catalog


In [0]:
%sql
create or replace table main.portfolio.revenue_mgmt_data_exp

WITH bookings as (
    select * 
    from main.portfolio.revenue_mgmt_data_exp
)
,
select DISTINCT 
res_id as booking_id, -- trim "res-" AND leading 0s from res_id
Fecha_Viaje as sail_date,
Fecha_Reserva as booked_date,
Lead_Time_Dias as DTS,
Noches_Estancia as sail_nights,
Barco as ship,
`Compañia` as brand, -- had to add backquotes with the non-traditional letter
Tipo_Ruta as product,
Suite_Type as cat_class,
Booking_Source as channel,
Package as booking_type,
Guest_Country as guest_country,
Cabinas_Reservadas as booked_cab,
Porcentaje_Ocupacion_Ciclo as load_factor,
Pasajeros_Reserva as pax_count,
Tripulacion as crew_count,
Ingreso_Total_Reserva_USD as ntr,
Cabinas_Totales_Barco as ship_cabin_capcity,
Gasto_Promedio_Diario_Huesped_USD as avg_daily_rate,
Puntuacion_Satisfaccion as guest_rating_feedback

from bookings

/*
-- checked if any duplicates exist, none found
    select count(*)
    from raw_0
    union ALL
    select count(*)
    from bookings
*/

num_affected_rows,num_inserted_rows


In [0]:
%sql
---- check the columns were renamed successfully
select * from main.portfolio.revenue_mgmt_data_exp
limit 10

booking_id,sail_date,booked_date,DTS,sail_nights,ship,brand,product,cat_class,channel,booking_type,guest_country,booked_cab,load_factor,pax_count,crew_count,ntr,ship_cabin_capcity,avg_daily_rate,guest_rating_feedback
RES-00033,2018-01-05,2017-09-27,100,9,Norwegian Prima,Norwegian Cruise Line,Fiordos,Oceanview,Web,All Inclusive,USA,1,81.33,2,1388,3889.26,1643,133.27,4.4
RES-00059,2018-01-05,2017-11-01,65,4,Disney Dream,Disney Cruise Line,Bahamas,Interior,B2B,All Inclusive,Spain,1,91.0,1,1458,1412.96,1250,187.64,4.9
RES-00074,2018-01-05,2017-09-25,102,4,Costa Smeralda,Costa Cruceros,Mediterráneo,Suite,Direct,All Inclusive,USA,1,83.26,2,1688,1702.16,2612,129.97,4.2
RES-00092,2018-01-05,2017-08-17,141,12,Queen Mary 2,Cunard Line,Transatlántico,Balcony,Web,Full Board,USA,1,86.18,2,1253,6397.68,1347,183.77,4.6
RES-00094,2018-01-05,2017-08-01,157,12,Queen Mary 2,Cunard Line,Transatlántico,Oceanview,Web,All Inclusive,Germany,2,86.18,2,1253,8623.44,1347,193.71,4.9
RES-00099,2018-01-05,2017-11-19,47,12,Queen Mary 2,Cunard Line,Transatlántico,Interior,Direct,Full Board + EXC,USA,2,86.18,4,1253,14238.24,1347,213.83,4.8
RES-00102,2018-01-05,2017-09-18,109,12,Queen Mary 2,Cunard Line,Transatlántico,Balcony,B2B,All Inclusive,France,1,86.18,2,1253,6719.28,1347,197.17,5.0
RES-00113,2018-01-05,2017-10-06,91,6,AIDAcosma,AIDA Cruises,Canarias,Balcony,Direct,All Inclusive,USA,1,82.05,2,1500,2482.8,2732,124.1,4.1
RES-00124,2018-01-06,2017-07-21,169,8,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Direct,All Inclusive,USA,1,87.85,1,2138,2525.76,2626,150.12,4.5
RES-00151,2018-01-06,2017-12-02,35,9,Norwegian Prima,Norwegian Cruise Line,Fiordos,Balcony,Web,All Inclusive,USA,1,85.25,2,1388,3858.48,1643,131.56,4.6


### Now we can manipulate the data and already there are a few ways to split the data:
    1. Executive-Level, Ship Performance & Guest Trends
    2. Booking-Level, Lowest Level of Granularity in this data
    3. Ship-Level, Brand/Ship Specific detail

In [0]:
%sql
create or replace table main.portfolio.raw_exec as 
---- executive level, will be used to create brand KPI tables and performance trends 
with data as (
select * from main.portfolio.revenue_mgmt_data_exp
)
select distinct
    ship,
    brand,
    booking_id,
    booked_date,
    sail_date, 
    DTS, 
    sail_nights, 
    product, 
    cat_class, 
    channel, 
    booking_type,
    guest_country,
    booked_cab,
    load_factor, 
    ntr as ntr_usd,
    pax_count,
    sail_nights * pax_count * booked_cab as pcd, 
    avg_daily_rate, 
    guest_rating_feedback 
from data

;
create or replace table main.portfolio.bookings as 
---- reservation level, data cleaned and standardized to be used for reporting
with data as (
select * from main.portfolio.revenue_mgmt_data_exp
)
select distinct
    booking_id, 
    sail_date, 
    DTS, 
    booked_date, 
    sail_nights, 
    ship, 
    brand, 
    product, 
    cat_class,
    channel, 
    booking_type,
    booked_cab, 
    load_factor, 
    pax_count,
    ntr as ntr_usd,
    avg_daily_rate 
from data
order by booking_id asc

;
create or replace table main.portfolio.ship_detail as
---- ship level, hardware specific info
with data as (
select * from main.portfolio.revenue_mgmt_data_exp
)
select distinct
ship, 
brand, 
sail_date,
pax_count, 
crew_count, 
ship_cabin_capcity 
from data
order by brand, ship, crew_count desc








num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from main.portfolio.raw_exec
limit 10
;
select * from main.portfolio.bookings
limit 10
;
select * from main.portfolio.ship_detail
limit 10 
;
DESCRIBE main.portfolio.raw_exec
;
DESCRIBE main.portfolio.bookings
;
DESCRIBE main.portfolio.ship_detail

ship,brand,booking_id,booked_date,sail_date,DTS,sail_nights,product,cat_class,channel,booking_type,guest_country,booked_cab,load_factor,ntr_usd,pax_count,pcd,avg_daily_rate,guest_rating_feedback
Queen Mary 2,Cunard Line,RES-00213,2017-10-04,2018-01-06,94,13,Transatlántico,Balcony,B2B,Full Board,USA,1,91.12,4717.05,1,13,197.25,4.9
Costa Smeralda,Costa Cruceros,RES-00307,2017-11-10,2018-01-07,58,7,Mediterráneo,Interior,Direct,Full Board,UK,1,89.73,2976.26,2,14,129.79,4.3
Costa Smeralda,Costa Cruceros,RES-00310,2017-08-14,2018-01-07,146,7,Mediterráneo,Oceanview,Web,All Inclusive,UK,1,89.73,2048.2,1,7,127.0,4.2
MSC World Europa,MSC Cruceros,RES-00482,2017-09-10,2018-01-13,125,6,Mediterráneo,Balcony,Web,Full Board,USA,1,76.98,1967.88,1,6,162.38,4.6
AIDAcosma,AIDA Cruises,RES-00713,2017-10-05,2018-01-14,101,7,Canarias,Suite,Direct,All Inclusive,USA,2,83.21,7743.12,6,42,129.16,4.2
Icon of the Seas,Royal Caribbean,RES-00745,2017-12-13,2018-01-19,37,7,Caribe,Interior,Direct,Full Board,Germany,1,78.82,2442.09,1,7,183.27,4.9
MSC World Europa,MSC Cruceros,RES-00849,2017-12-18,2018-01-20,33,7,Mediterráneo,Oceanview,B2B,All Inclusive,Germany,1,77.52,3287.06,2,14,151.99,4.4
Disney Dream,Disney Cruise Line,RES-00896,2017-10-12,2018-01-20,100,5,Bahamas,Suite,B2B,All Inclusive + EXC,USA,2,81.45,3559.3,2,10,190.33,4.8
Costa Smeralda,Costa Cruceros,RES-01030,2017-11-09,2018-01-21,73,6,Mediterráneo,Oceanview,Web,Full Board + EXC,Mexico,2,73.56,5477.76,4,24,145.44,4.0
Disney Dream,Disney Cruise Line,RES-01606,2017-09-16,2018-02-03,140,5,Bahamas,Oceanview,Direct,Full Board,USA,1,90.58,1782.25,1,5,190.85,5.0


booking_id,sail_date,DTS,booked_date,sail_nights,ship,brand,product,cat_class,channel,booking_type,booked_cab,load_factor,ntr_usd,avg_daily_rate
RES-00001,2018-01-05,140,2017-08-18,6,MSC World Europa,MSC Cruceros,Mediterráneo,Balcony,Direct,All Inclusive,1,87.12,2762.28,147.39
RES-00002,2018-01-05,113,2017-09-14,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,B2B,All Inclusive + EXC,1,87.12,1863.06,144.91
RES-00003,2018-01-05,65,2017-11-01,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Web,All Inclusive,1,87.12,2760.48,147.24
RES-00004,2018-01-05,66,2017-10-31,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,All Inclusive + EXC,2,87.12,3883.68,158.04
RES-00005,2018-01-05,110,2017-09-17,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Direct,All Inclusive + EXC,1,87.12,2723.28,144.14
RES-00006,2018-01-05,45,2017-11-21,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,All Inclusive,1,87.12,2753.52,146.66
RES-00007,2018-01-05,55,2017-11-11,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,Web,Full Board,2,87.12,3671.4,140.35
RES-00008,2018-01-05,114,2017-09-13,6,MSC World Europa,MSC Cruceros,Mediterráneo,Interior,B2B,All Inclusive,1,87.12,2848.08,154.54
RES-00009,2018-01-05,5,2017-12-31,6,MSC World Europa,MSC Cruceros,Mediterráneo,Balcony,Direct,Full Board,1,87.12,2685.96,141.03
RES-00010,2018-01-05,57,2017-11-09,6,MSC World Europa,MSC Cruceros,Mediterráneo,Oceanview,Web,All Inclusive,1,87.12,2741.52,145.66


ship,brand,sail_date,pax_count,crew_count,ship_cabin_capcity
AIDAcosma,AIDA Cruises,2018-04-15,1,1500,2732
AIDAcosma,AIDA Cruises,2018-05-12,2,1500,2732
AIDAcosma,AIDA Cruises,2018-05-27,2,1500,2732
AIDAcosma,AIDA Cruises,2018-06-16,1,1500,2732
AIDAcosma,AIDA Cruises,2018-07-13,1,1500,2732
AIDAcosma,AIDA Cruises,2018-08-05,1,1500,2732
AIDAcosma,AIDA Cruises,2018-09-14,2,1500,2732
AIDAcosma,AIDA Cruises,2018-12-21,2,1500,2732
AIDAcosma,AIDA Cruises,2019-01-04,1,1500,2732
AIDAcosma,AIDA Cruises,2019-01-06,2,1500,2732


col_name,data_type,comment
ship,string,null
brand,string,null
booking_id,string,null
booked_date,date,null
sail_date,date,null
DTS,bigint,null
sail_nights,bigint,null
product,string,null
cat_class,string,null
channel,string,null


col_name,data_type,comment
booking_id,string,null
sail_date,date,null
DTS,bigint,null
booked_date,date,null
sail_nights,bigint,null
ship,string,null
brand,string,null
product,string,null
cat_class,string,null
channel,string,null


col_name,data_type,comment
ship,string,null
brand,string,null
sail_date,date,null
pax_count,bigint,null
crew_count,bigint,null
ship_cabin_capcity,bigint,null


## Data Transformations



**Transformation Detail** \
    1. **Create specfic source** used for performance data querying using basic _DBA FUNCTIONS_ \
        --- best practice when used with jobs for continuous data refresh \
    2. Used _EXTRACT_ **time-based attritbutes** using booking date and sail date columns \
    3. Created columns to deduce **lead time** (weeks/months to sail date) using _FLOOR_ \
    4. **Created classification attributes** using _CASE WHEN_  
    5. **Created value-based KPIs** with _aggregate functions_ 

## Weekly Performance & High-Level Brand Insights

In [0]:
%sql
create or replace table main.portfolio.exec_silver as

with data as (
    select * from main.portfolio.raw_exec
    --limit 10
)
---
select distinct
    brand, -- 1,2,3
    extract(year from booked_date) booked_year,
    extract(month from booked_date) booked_month,
    ship, -- 1,2,3
    --sail_date, -- 1,2,3
    extract(year from sail_date) as sail_year,
    extract(month from sail_date) as sail_month,
    case 
        when extract(month from sail_date) in (1,2,3)
            then '1'
        when extract(month from sail_date) in (4,5,6)
            then '2'
        when extract(month from sail_date) in (7,8,9)
            then '3'
        when extract(month from sail_date) in (10,11,12)
            then '4'
        else 'check'
    end as sail_qtr,
    DTS, -- 1,2
    floor(DTS / 7) as WTS, -- weeks to sail
    floor(DTS / 30) as MTS, -- months to sail
    sail_nights, -- 1,2
    case 
        when sail_nights in (1,2,3,4,5)
            then 'Shorts'
        when sail_nights in (6,7,8,9)
            then 'Getaway'
        when sail_nights > 9
            then 'Expedition'
        else 'check'
    end as sail_night_group,
    product, --1,2
    cat_class, -- 1,2
    channel, -- 1,2
    booking_type, -- 1,2
    guest_country, -- 1
    sum(booked_cab) as booked_cab,
    sum(ntr_usd) as ntr_usd, -- 1,2
    sum(pcd) pcd,
    count(booking_id) bookings, -- 1,2
    avg(load_factor) as load_factor, -- already percent but can't be used at row level, must be recalculated.
    -- we can join back capacity from our ship_detail table later, would be a dead/repeating rows
    avg(avg_daily_rate) as avg_APD, -- 1,2
    avg(guest_rating_feedback) as guest_rating_feedback -- 1

from data

group by
brand, -- 1,2,3
ship, -- 1,2,3
sail_date, -- 1,2,3
sail_year,
sail_month,
sail_qtr,
booked_year,
booked_month,
DTS, -- 1,2
WTS, -- weeks to sail
MTS, -- months to sail
sail_nights, -- 1,2
sail_night_group,
product, --1,2
cat_class, -- 1,2
channel, -- 1,2
booking_type, -- 1,2
guest_country

order by brand, ship, sail_year, sail_month asc, booked_year asc, booked_month asc, MTS asc
;
select * from main.portfolio.exec_silver
limit 10
;
DESCRIBE main.portfolio.exec_silver


num_affected_rows,num_inserted_rows


brand,booked_year,booked_month,ship,sail_year,sail_month,sail_qtr,DTS,WTS,MTS,sail_nights,sail_night_group,product,cat_class,channel,booking_type,guest_country,booked_cab,ntr_usd,pcd,bookings,load_factor,avg_APD,guest_rating_feedback
AIDA Cruises,2017,7,AIDAcosma,2018,1,1,179,25,5,7,Getaway,Canarias,Interior,Direct,All Inclusive,USA,2,5490.52,56,1,82.53,113.29,4.5
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,149,21,4,7,Getaway,Canarias,Oceanview,Direct,All Inclusive,Spain,2,3969.0,28,1,83.21,117.9,4.3
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,132,18,4,6,Getaway,Canarias,Interior,B2B,All Inclusive + EXC,USA,1,2442.0,12,1,89.05,120.7,4.6
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,133,19,4,6,Getaway,Canarias,Balcony,B2B,All Inclusive,UK,1,1645.92,6,1,90.06,108.72,4.2
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,139,19,4,6,Getaway,Canarias,Balcony,B2B,All Inclusive,Germany,1,2402.76,12,1,82.05,117.43,4.4
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,130,18,4,6,Getaway,Canarias,Balcony,B2B,All Inclusive,USA,1,2418.0,12,1,90.06,118.7,4.2
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,145,20,4,6,Getaway,Canarias,Oceanview,B2B,All Inclusive + EXC,USA,1,3122.82,18,1,90.06,118.29,4.5
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,169,24,5,7,Getaway,Canarias,Oceanview,Direct,All Inclusive + EXC,UK,1,2838.5,14,1,88.73,119.95,4.2
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,151,21,5,6,Getaway,Canarias,Oceanview,Web,Full Board,Spain,1,1694.22,6,1,90.06,116.77,4.3
AIDA Cruises,2017,8,AIDAcosma,2018,1,1,167,23,5,8,Getaway,Canarias,Balcony,B2B,Full Board + EXC,Spain,1,2342.4,8,1,78.29,127.2,4.3


col_name,data_type,comment
brand,string,null
booked_year,int,null
booked_month,int,null
ship,string,null
sail_year,int,null
sail_month,int,null
sail_qtr,string,null
DTS,bigint,null
WTS,bigint,null
MTS,bigint,null


#### Example Use Case: High-Level Performance KPIs
Now with the executive view table cleaned and prepped for reporting, what kind of insights can we source from here. In this example we can summarize overall volume and rate performance with some cruise-specific indicator formulas: \
    - Load Factor \
    - Average Revenue Per Day Per Guest (APD) \
    - Average Daily Rate (Price) \
    - Booking Revenue (NTR) \
    - Average Cruise Days per Passenger (PCD)


In [0]:
%sql
---Brand Booking Volumes by Brand only and brand and ship for extended view
with data as (
    select * from main.portfolio.exec_silver
)

select 
    brand,
    --sail_year,
    sail_qtr,
    sum(bookings) bookings,
    sum(ntr_usd) ntr_usd,
    sum(booked_cab) booked_cab

from data

where sail_year = 2018 --only 2018 sailings in here

group by brand, sail_qtr

order by 
brand,
sail_qtr

;
select 
brand,
sail_qtr,
sum(bookings) bookings,
round(sum(ntr_usd) / 1000000, 2) as ntr_usd_in_M, --simplify revenue to millions
round(sum(pcd) / 1000,1) as pcd_in_K,
round(sum(ntr_usd) / sum(pcd),2) as avg_daily_rate_calc -- this is the real APD formula (Total Revenue / (Total Passengers * Cruise Days))
--round(avg(avg_apd),2) as avg_APD_per_bkg --- not the correct way to for summarization of average rate, never take an average of an average

from main.portfolio.exec_silver

where sail_year = 2018 --only 2018 sailings in here

group by brand, sail_qtr

order by brand, sail_qtr





brand,sail_qtr,bookings,ntr_usd,booked_cab
AIDA Cruises,1,570,1615172.8300000029,666
AIDA Cruises,2,585,1686063.1399999997,664
AIDA Cruises,3,600,1773984.299999999,692
AIDA Cruises,4,585,1584041.0399999998,660
Celebrity Cruises,1,570,2718646.1800000006,667
Celebrity Cruises,2,585,3002111.8600000003,665
Celebrity Cruises,3,600,3034958.540000002,695
Celebrity Cruises,4,585,2795463.940000001,669
Costa Cruceros,1,570,1603257.560000001,658
Costa Cruceros,2,585,1797014.7200000002,681


brand,sail_qtr,bookings,ntr_usd_in_M,pcd_in_K,avg_daily_rate_calc,avg_APD
AIDA Cruises,1,570,1.62,8.9,182.03,123.14
AIDA Cruises,2,585,1.69,8.4,201.88,132.75
AIDA Cruises,3,600,1.77,8.8,201.52,134.17
AIDA Cruises,4,585,1.58,7.8,202.38,129.03
Celebrity Cruises,1,570,2.72,12.5,216.69,166.75
Celebrity Cruises,2,585,3.0,12.6,238.79,179.31
Celebrity Cruises,3,600,3.03,12.6,240.13,181.85
Celebrity Cruises,4,585,2.8,12.0,233.71,174.74
Costa Cruceros,1,570,1.6,8.3,192.51,132.62
Costa Cruceros,2,585,1.8,8.8,204.79,142.95


#### Example Use Case: Top Brand Competitor Trends
Let us take a look at the _overall guest ratings of the [brands_](url) and the _average rate of each cabin category_ for a few of those top competitors.

In [0]:
%sql
with data as (
    select * from main.portfolio.exec_silver
)

select * from (
 select brand, 
 ship, 
 round(avg(guest_rating_feedback),1) as brand_score

from data
group by brand, ship
)

order by brand_score desc

;
with data as (
    select * from main.portfolio.exec_silver
)
select 
brand,
cat_class, 
---sum(ntr_usd) ntr, 
---sum(pcd) pcd, 
round(sum(ntr_usd) / sum(pcd),2) apd_calc 
from data
where brand LIKE '%MSC%' OR brand LIKE '%Royal%' OR brand LIKE '%Disney%'

group by brand, cat_class


brand,ship,brand_score
Disney Cruise Line,Disney Dream,4.9
Cunard Line,Queen Mary 2,4.8
Royal Caribbean,Icon of the Seas,4.8
Celebrity Cruises,Celebrity Beyond,4.7
MSC Cruceros,MSC World Europa,4.5
Norwegian Cruise Line,Norwegian Prima,4.5
AIDA Cruises,AIDAcosma,4.3
Costa Cruceros,Costa Smeralda,4.2


brand,cat_class,apd_calc
Disney Cruise Line,Balcony,255.26
Disney Cruise Line,Suite,258.01
Disney Cruise Line,Interior,254.57
Disney Cruise Line,Oceanview,255.99
MSC Cruceros,Balcony,223.88
MSC Cruceros,Interior,225.76
MSC Cruceros,Oceanview,221.51
MSC Cruceros,Suite,222.44
Royal Caribbean,Interior,242.58
Royal Caribbean,Balcony,241.08


## Booking Level Analysis

For this analysis, we will look at the following from the bookings level:
1. Booking Windows by Brand (Bookings by MTS)
2. Most Popular Products by Brand
3. Product Shares of Cunard Cruises
4. Booking Channel Shares
5. Historical Channel Trends since 2018


In [0]:
%sql
create or replace table main.portfolio.bookings_silver as

with data as (
    select * from main.portfolio.bookings
    --limit 10
)

select distinct
    brand, -- 1,2,3
    extract(year from booked_date) booked_year,
    extract(month from booked_date) booked_month,
    ship, -- 1,2,3
    sail_date, -- 1,2,3
    extract(year from sail_date) as sail_year,
    extract(month from sail_date) as sail_month,
    floor(DTS / 30) as MTS, -- months to sail
    sail_nights, -- 1,2
    case 
        when sail_nights in (1,2,3,4,5)
            then 'Shorts'
        when sail_nights in (6,7,8,9)
            then 'Getaway'
        when sail_nights > 9
            then 'Expedition'
        else 'check'
    end as sail_night_group,
    product, --1,2
    cat_class,
    channel, 
    booking_type,
    sum(pax_count) pax,
    sum(booked_cab) as booked_cab, 
    avg(load_factor) avg_LF, 
    sum(ntr_usd) as ntr_usd,
    avg(avg_daily_rate) avg_ADR 

    from data

    group by
    brand,
    booked_year,
    booked_month,
    ship,
    sail_date,
    sail_year,
    sail_month,
    MTS, 
    sail_night_group,
    product, 
    sail_nights, 
    cat_class,
    channel, 
    booking_type

    ;
    select * from main.portfolio.bookings_silver
    limit 10
    ;
    describe main.portfolio.bookings_silver

num_affected_rows,num_inserted_rows


brand,booked_year,booked_month,ship,sail_date,sail_year,sail_month,MTS,sail_nights,sail_night_group,product,cat_class,channel,booking_type,pax,booked_cab,avg_LF,ntr_usd,avg_ADR
MSC Cruceros,2017,9,MSC World Europa,2018-01-05,2018,1,3,6,Getaway,Mediterráneo,Interior,B2B,All Inclusive + EXC,1,1,87.12,1863.06,144.91
Royal Caribbean,2017,10,Icon of the Seas,2018-01-05,2018,1,2,7,Getaway,Caribe,Interior,Direct,All Inclusive,2,1,80.76,3825.08,190.42
Norwegian Cruise Line,2017,10,Norwegian Prima,2018-01-05,2018,1,3,9,Getaway,Fiordos,Oceanview,Web,Full Board,1,1,81.33,2729.88,137.72
Disney Cruise Line,2017,10,Disney Dream,2018-01-05,2018,1,2,4,Shorts,Bahamas,Suite,B2B,All Inclusive + EXC,2,1,91.0,2163.04,187.58
Disney Cruise Line,2017,9,Disney Dream,2018-01-05,2018,1,3,4,Shorts,Bahamas,Balcony,Direct,All Inclusive,1,1,91.0,1396.8,183.6
Costa Cruceros,2017,10,Costa Smeralda,2018-01-05,2018,1,2,4,Shorts,Mediterráneo,Balcony,Web,All Inclusive,1,1,83.26,1191.6,132.3
Celebrity Cruises,2017,12,Celebrity Beyond,2018-01-05,2018,1,1,9,Getaway,Alaska,Interior,B2B,Full Board + EXC,1,1,91.07,2921.31,158.99
Cunard Line,2017,10,Queen Mary 2,2018-01-05,2018,1,2,12,Expedition,Transatlántico,Oceanview,Direct,Full Board + EXC,2,1,86.18,6632.88,193.57
Royal Caribbean,2017,9,Icon of the Seas,2018-01-06,2018,1,3,5,Shorts,Caribe,Balcony,Web,Full Board,2,1,87.88,2577.7,174.97
Costa Cruceros,2017,8,Costa Smeralda,2018-01-06,2018,1,4,8,Getaway,Mediterráneo,Interior,Web,Full Board + EXC,1,1,83.4,2441.2,139.55


col_name,data_type,comment
brand,string,null
booked_year,int,null
booked_month,int,null
ship,string,null
sail_date,date,null
sail_year,int,null
sail_month,int,null
MTS,bigint,null
sail_nights,bigint,null
sail_night_group,string,null


In [0]:
%sql
with data as (
    select * from main.portfolio.bookings_silver
    --limit 10
)
--- 
select 
brand,
MTS,
round((pax / brand_totals) * 100, 1) as bkg_share

from
    (
        select distinct
        a.brand,
        a.MTS,
        sum(a.pax) pax,
        b.brand_totals brand_totals

        from data a

        left join (select distinct brand, sum(pax) brand_totals from data group by brand order by brand) b
        on a.brand = b.brand

        group by a.brand, a.MTS, b.brand_totals
    )
order by brand, MTS


brand,MTS,bkg_share
AIDA Cruises,0,4.1
AIDA Cruises,1,12.6
AIDA Cruises,2,24.4
AIDA Cruises,3,27.5
AIDA Cruises,4,19.1
AIDA Cruises,5,9.3
AIDA Cruises,6,2.6
AIDA Cruises,7,0.3
AIDA Cruises,8,0.0
Celebrity Cruises,0,3.7


In [0]:
%sql
with data as (
    select * from main.portfolio.bookings_silver
    --limit 10
)

select brand, product, round(pax,2) as pax_in_K

from (
    select distinct
    brand, 
    product, 
    sum(pax) / 1000 pax,
    rank() over (partition by brand, product order by sum(pax) desc) as rank

    from data
    group by brand, product
)

where rank = 1

order by brand


brand,product,pax_in_K
AIDA Cruises,Canarias,16.9
Celebrity Cruises,Alaska,16.87
Costa Cruceros,Mediterráneo,16.89
Cunard Line,Transatlántico,16.8
Disney Cruise Line,Bahamas,16.81
MSC Cruceros,Mediterráneo,16.87
Norwegian Cruise Line,Fiordos,16.95
Royal Caribbean,Caribe,16.95


In [0]:
%sql
with data as (
    select * from main.portfolio.bookings_silver
    --limit 10
)

select brand, booking_type, sum(pax) pax

from data

where brand like '%Cunard%'

group by brand, booking_type

order by pax desc


brand,booking_type,pax
Cunard Line,All Inclusive,6759
Cunard Line,Full Board,4078
Cunard Line,All Inclusive + EXC,3413
Cunard Line,Full Board + EXC,2551


In [0]:
%sql
with data as (
    select * from main.portfolio.bookings_silver
    --limit 10
)

select distinct
sail_year,
channel,
sum(pax) over (partition by sail_year,channel) as pax

from data 

order by sail_year, pax desc


sail_year,channel,pax
2018,B2B,14784
2018,Web,11362
2018,Direct,6667
2019,B2B,14623
2019,Web,11552
2019,Direct,6622
2020,B2B,14758
2020,Web,11561
2020,Direct,6693
2021,B2B,14749


## Brand Hardware Detail, Ship-Level Insights

#### Example Use-Case: Guest-to-Crew Ratio by Brand


In [0]:
%sql
with data as (
    select distinct * from main.portfolio.ship_detail
    --limit 10
)

select 
    *,
    round(ship_cabin_capcity / crew_count,1) as cabin_to_crew_ratio 
from (

    select * except(sail_date, pax_count) 
    from data 
    group by brand, ship, crew_count, ship_cabin_capcity 
    order by brand, ship
)

ship,brand,crew_count,ship_cabin_capcity,cabin_to_crew_ratio
AIDAcosma,AIDA Cruises,1500,2732,1.8
Celebrity Beyond,Celebrity Cruises,1416,1646,1.2
Costa Smeralda,Costa Cruceros,1688,2612,1.5
Queen Mary 2,Cunard Line,1253,1347,1.1
Disney Dream,Disney Cruise Line,1458,1250,0.9
MSC World Europa,MSC Cruceros,2138,2626,1.2
Norwegian Prima,Norwegian Cruise Line,1388,1643,1.2
Icon of the Seas,Royal Caribbean,2350,2805,1.2
